# Pandas Intermediate

This notebook covers intermediate-level Pandas techniques for data manipulation and analysis.

## What You Will Learn:
- MultiIndex and hierarchical indexing
- Advanced indexing with query, isin, and between
- Advanced GroupBy operations (agg, transform, filter, apply)
- Merging, joining, and concatenating DataFrames
- Pivot tables and cross-tabulation
- Working with text data using the str accessor
- Working with date and time data
- Window functions (rolling, expanding, ewm)
- Working with categorical data
- Advanced missing data handling (interpolation)

## 1. MultiIndex / Hierarchical Indexing

MultiIndex allows you to work with higher-dimensional data in a 2D DataFrame. It enables grouping data by multiple keys.

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# Create a MultiIndex from arrays
arrays = [
    ['A', 'A', 'A', 'A', 'B', 'B', 'B', 'B'],
    ['one', 'one', 'two', 'two', 'one', 'one', 'two', 'two']
]
index = pd.MultiIndex.from_arrays(arrays, names=['First', 'Second'])

df_multi = pd.DataFrame(np.random.randn(8, 2), index=index, columns=['X', 'Y'])
print(df_multi)

In [ ]:
# Create MultiIndex from tuples
tuples = [
    ('A', 'one'), ('A', 'one'), ('A', 'two'), ('A', 'two'),
    ('B', 'one'), ('B', 'one'), ('B', 'two'), ('B', 'two')
]
index2 = pd.MultiIndex.from_tuples(tuples, names=['Level1', 'Level2'])
df_multi2 = pd.DataFrame(np.random.randn(8, 3), index=index2, columns=['Col1', 'Col2', 'Col3'])
print(df_multi2)

In [ ]:
# Create MultiIndex from a product
index3 = pd.MultiIndex.from_product(
    [['A', 'B', 'C'], ['X', 'Y', 'Z']],
    names=['Outer', 'Inner']
)
df_multi3 = pd.DataFrame(np.random.randn(9, 2), index=index3, columns=['Data1', 'Data2'])
print(df_multi3)

In [ ]:
# Selecting from MultiIndex
print("Select level 'A':")
print(df_multi.xs('A', level='First'))

print("\nSelect ('A', 'one'):")
print(df_multi.loc[('A', 'one')])

print("\nSelect all 'one' at second level:")
print(df_multi.xs('one', level='Second'))

print("\nSlice using loc:")
print(df_multi.loc[pd.IndexSlice[:, 'one'], :])

In [ ]:
# Stack and unstack
print("Original:")
print(df_multi)

print("\nStacked:")
print(df_multi.stack())

print("\nUnstacked:")
print(df_multi.unstack())

In [ ]:
# Reset and set index
print("Reset index:")
df_reset = df_multi.reset_index()
print(df_reset)

print("\nSet index back:")
df_set = df_reset.set_index(['First', 'Second'])
print(df_set)

print("\nSort index:")
print(df_multi.sort_index())

## 2. Advanced Indexing

Beyond `loc` and `iloc`, Pandas provides several powerful indexing methods.

In [ ]:
# Create sample data
df_adv = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace'],
    'Age': [25, 30, 35, 28, 32, 45, 38],
    'Score': [85, 92, 78, 95, 88, 72, 81],
    'City': ['NYC', 'LA', 'Tokyo', 'Paris', 'Sydney', 'London', 'Berlin']
})
print(df_adv)

In [ ]:
# query() method
print("Age > 30:")
print(df_adv.query('Age > 30'))

print("\nAge > 30 and Score > 80:")
print(df_adv.query('Age > 30 and Score > 80'))

print("\nCity in list:")
print(df_adv.query('City in ["NYC", "Paris", "London"]'))

print("\nUsing variables in query:")
min_age = 30
print(df_adv.query('Age > @min_age'))

In [ ]:
# isin() method
print("Age in [25, 35, 45]:")
print(df_adv[df_adv['Age'].isin([25, 35, 45])])

print("\nCity in list:")
print(df_adv[df_adv['City'].isin(['NYC', 'LA', 'Tokyo'])])

print("\nMultiple columns with isin:")
print(df_adv[df_adv[['Name', 'City']].isin({'Name': ['Alice'], 'City': ['NYC']}).all(axis=1)])

In [ ]:
# between() method
print("Age between 28 and 35:")
print(df_adv[df_adv['Age'].between(28, 35)])

print("\nScore between 80 and 90:")
print(df_adv[df_adv['Score'].between(80, 90)])

print("\nCombined between:")
print(df_adv[df_adv['Age'].between(25, 40) & df_adv['Score'].between(80, 90)])

In [ ]:
# where() and mask() methods
print("where() - keep values where condition is True, replace with NaN otherwise:")
print(df_adv['Score'].where(df_adv['Score'] >= 85))

print("\nmask() - replace values where condition is True:")
print(df_adv['Score'].mask(df_adv['Score'] < 85, 0))

print("\nwhere with replacement:")
print(df_adv['Score'].where(df_adv['Score'] >= 85, 'Low'))

## 3. Advanced GroupBy Operations

In [ ]:
# Create a more complex dataset
np.random.seed(42)
df_group = pd.DataFrame({
    'Product': np.random.choice(['A', 'B', 'C'], size=20),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], size=20),
    'Sales': np.random.randint(100, 500, size=20),
    'Quantity': np.random.randint(1, 50, size=20),
    'Discount': np.random.uniform(0, 0.2, size=20).round(2)
})
print(df_group.head(10))

In [ ]:
# agg() - multiple aggregations
print("Multiple aggregations on Sales:")
print(df_group.groupby('Product')['Sales'].agg(['sum', 'mean', 'min', 'max', 'std']))

print("\nNamed aggregations:")
print(df_group.groupby('Product').agg(
    Total_Sales=('Sales', 'sum'),
    Avg_Sales=('Sales', 'mean'),
    Max_Discount=('Discount', 'max'),
    Total_Quantity=('Quantity', 'sum')
))

In [ ]:
# agg() with custom functions
def range_func(x):
    return x.max() - x.min()

print("Custom aggregation:")
print(df_group.groupby('Product').agg({
    'Sales': ['sum', 'mean', range_func],
    'Quantity': ['sum', 'count'],
    'Discount': ['mean', 'std']
}))

In [ ]:
# transform() - returns a like-indexed Series
df_group['Sales_Rank'] = df_group.groupby('Product')['Sales'].rank(ascending=False)
df_group['Sales_ZScore'] = df_group.groupby('Product')['Sales'].transform(
    lambda x: (x - x.mean()) / x.std()
)
df_group['Sales_Percentile'] = df_group.groupby('Product')['Sales'].transform(
    lambda x: x / x.sum() * 100
)
print(df_group.head(10))

In [ ]:
# filter() - filter groups based on a condition
print("Groups with more than 3 rows:")
print(df_group.groupby('Product').filter(lambda x: len(x) > 3))

print("\nProducts with average Sales > 250:")
print(df_group.groupby('Product').filter(lambda x: x['Sales'].mean() > 250))

In [ ]:
# apply() - apply a function to each group
print("Apply custom function to each group:")
result = df_group.groupby('Product').apply(
    lambda x: pd.Series({
        'Total_Sales': x['Sales'].sum(),
        'Avg_Quantity': x['Quantity'].mean(),
        'Max_Discount': x['Discount'].max()
    })
)
print(result)

In [ ]:
# GroupBy with multiple keys
print("Group by Product and Region:")
grouped_multi = df_group.groupby(['Product', 'Region'])
print(grouped_multi['Sales'].agg(['sum', 'mean', 'count']))

print("\nGroup by Product, then by Region:")
for product, group in df_group.groupby('Product'):
    print(f"\nProduct: {product}")
    print(group.groupby('Region')['Sales'].sum())

## 4. Merging, Joining, and Concatenating

In [ ]:
# Create sample DataFrames
df1 = pd.DataFrame({
    'ID': [1, 2, 3, 4, 5],
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'Age': [25, 30, 35, 28, 32]
})

df2 = pd.DataFrame({
    'ID': [1, 2, 3, 4, 6],
    'City': ['NYC', 'LA', 'Tokyo', 'Paris', 'Sydney'],
    'Salary': [75000, 80000, 90000, 65000, 85000]
})

print("DataFrame 1:")
print(df1)
print("\nDataFrame 2:")
print(df2)

In [ ]:
# Merge - SQL-style joins
print("Inner join (default):")
print(pd.merge(df1, df2, on='ID'))

print("\nLeft join:")
print(pd.merge(df1, df2, on='ID', how='left'))

print("\nRight join:")
print(pd.merge(df1, df2, on='ID', how='right'))

print("\nOuter join:")
print(pd.merge(df1, df2, on='ID', how='outer'))

In [ ]:
# Merge with different column names
df3 = pd.DataFrame({
    'EmployeeID': [1, 2, 3, 4],
    'Department': ['Eng', 'Mkt', 'Sales', 'HR']
})

print("Merge with different column names:")
print(pd.merge(df1, df3, left_on='ID', right_on='EmployeeID', how='left'))

# Merge on index
df4 = pd.DataFrame({'Score': [85, 90, 78, 95]}, index=[1, 2, 3, 4])
print("\nMerge on index:")
print(pd.merge(df1, df4, left_on='ID', right_index=True, how='left'))

In [ ]:
# Join - simpler syntax for index-based joins
df_left = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]}, index=['x', 'y', 'z'])
df_right = pd.DataFrame({'C': [7, 8, 9], 'D': [10, 11, 12]}, index=['x', 'y', 'w'])

print("Left DataFrame:")
print(df_left)
print("\nRight DataFrame:")
print(df_right)

print("\nJoin (left):")
print(df_left.join(df_right, how='left'))

print("\nJoin (outer):")
print(df_left.join(df_right, how='outer'))

In [ ]:
# Concat - stack DataFrames
df_a = pd.DataFrame({'Key': ['A', 'B'], 'Val': [1, 2]})
df_b = pd.DataFrame({'Key': ['C', 'D'], 'Val': [3, 4]})

print("Vertical concatenation:")
print(pd.concat([df_a, df_b], axis=0))

df_c = pd.DataFrame({'Key': ['A', 'B'], 'Val2': [5, 6]})
print("\nHorizontal concatenation:")
print(pd.concat([df_a, df_c], axis=1))

print("\nConcatenation with keys:")
print(pd.concat([df_a, df_b], keys=['First', 'Second']))

In [ ]:
# Update - modify values in place
df_orig = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]}, index=['x', 'y', 'z'])
df_update = pd.DataFrame({'A': [10, 20]}, index=['x', 'y'])

print("Original:")
print(df_orig)
print("\nUpdate:")
print(df_update)

df_orig.update(df_update)
print("\nAfter update:")
print(df_orig)

## 5. Pivot Tables and Cross-Tabulation

In [ ]:
# Create a sales dataset for pivot tables
np.random.seed(42)
sales = pd.DataFrame({
    'Date': pd.date_range('2024-01-01', periods=12, freq='M'),
    'Product': ['A', 'B', 'C'] * 4,
    'Region': ['North', 'South', 'East', 'West'] * 3,
    'Sales': np.random.randint(100, 500, size=12),
    'Quantity': np.random.randint(10, 50, size=12)
})
print(sales)

In [ ]:
# Basic pivot table
print("Pivot table - Sales by Product and Region:")
print(pd.pivot_table(sales, values='Sales', index='Product', columns='Region', aggfunc='sum'))

print("\nWith fill_value:")
print(pd.pivot_table(sales, values='Sales', index='Product', columns='Region', aggfunc='sum', fill_value=0))

print("\nMultiple aggregation functions:")
print(pd.pivot_table(sales, values='Sales', index='Product', aggfunc=['sum', 'mean', 'std']))

In [ ]:
# Advanced pivot table
print("Multiple values:")
print(pd.pivot_table(sales, values=['Sales', 'Quantity'], index='Product', columns='Region', aggfunc='sum'))

print("\nWith margins:")
print(pd.pivot_table(sales, values='Sales', index='Product', columns='Region', aggfunc='sum', margins=True))

print("\nPivot with Grouper:")
print(pd.pivot_table(sales, values='Sales', index=pd.Grouper(key='Date', freq='Q'), columns='Product', aggfunc='sum'))

In [ ]:
# Cross-tabulation with more options
np.random.seed(42)
survey = pd.DataFrame({
    'Gender': np.random.choice(['Male', 'Female'], size=100),
    'AgeGroup': np.random.choice(['18-25', '26-35', '36-50'], size=100),
    'Satisfaction': np.random.choice(['Low', 'Medium', 'High'], size=100),
    'Score': np.random.randint(1, 10, size=100)
})

print("Basic cross-tab:")
print(pd.crosstab(survey['Gender'], survey['Satisfaction']))

print("\nWith values and aggfunc:")
print(pd.crosstab(survey['Gender'], survey['AgeGroup'], values=survey['Score'], aggfunc='mean'))

print("\nWith margins:")
print(pd.crosstab(survey['Gender'], survey['Satisfaction'], margins=True, margins_name='Total'))

print("\nNormalized:")
print(pd.crosstab(survey['Gender'], survey['Satisfaction'], normalize='index'))

In [ ]:
# Multiple cross-tabs
print("Three-way cross-tab:")
ct = pd.crosstab(
    [survey['Gender'], survey['AgeGroup']],
    survey['Satisfaction'],
    margins=True
)
print(ct)

## 6. Working with Text Data (str accessor)

In [ ]:
# Create sample text data
df_text = pd.DataFrame({
    'Name': ['alice smith', 'BOB JOHNSON', 'Charlie Brown', 'diana prince', 'Eve Adams'],
    'Email': ['alice@example.com', 'bob@test.org', 'charlie@gmail.com', 'diana@company.net', 'eve@site.io'],
    'Phone': ['555-1234', '555-5678', '555-9012', '555-3456', '555-7890'],
    'Address': ['123 Main St, NYC', '456 Oak Ave, LA', '789 Pine Rd, Chicago', '321 Elm St, Houston', '654 Maple Dr, Phoenix']
})
print(df_text)

In [ ]:
# String methods
print("Upper case:")
print(df_text['Name'].str.upper())

print("\nTitle case:")
print(df_text['Name'].str.title())

print("\nLength:")
print(df_text['Name'].str.len())

print("\nContains '@':")
print(df_text['Email'].str.contains('@'))

print("\nStarts with 'a':")
print(df_text['Name'].str.startswith('a'))

In [ ]:
# String splitting and extraction
print("Split Name:")
print(df_text['Name'].str.split(expand=True))

print("\nExtract email domain:")
print(df_text['Email'].str.extract(r'@(\w+\.\w+)'))

print("\nExtract phone number:")
print(df_text['Phone'].str.extract(r'(\d{3})-(\d{4})'))

print("\nReplace in address:")
print(df_text['Address'].str.replace('St', 'Street', regex=False))

In [ ]:
# String padding and formatting
df_id = pd.DataFrame({'ID': ['1', '23', '456', '7890']})
print("Original:")
print(df_id['ID'])

print("\nZero-padded to 5 digits:")
print(df_id['ID'].str.zfill(5))

print("\nRight-padded:")
print(df_id['ID'].str.ljust(5, '0'))

print("\nStrip whitespace:")
df_ws = pd.DataFrame({'Text': ['  hello  ', ' world ', '  pandas  ']})

In [ ]:
print(df_ws['Text'].str.strip())
print(df_ws['Text'].str.strip().str.upper())

## 7. Working with Date and Time

In [ ]:
# Create date range
dates = pd.date_range('2024-01-01', periods=10, freq='D')
print("Date range:")
print(dates)

# Different frequencies
print("\nWeekly:")
print(pd.date_range('2024-01-01', periods=5, freq='W'))

print("\nMonthly:")
print(pd.date_range('2024-01-01', periods=5, freq='M'))

print("\nBusiness days:")
print(pd.date_range('2024-01-01', periods=10, freq='B'))

print("\nHourly:")
print(pd.date_range('2024-01-01', periods=5, freq='H'))

In [ ]:
# Convert string to datetime
df_dates = pd.DataFrame({
    'Date': ['2024-01-15', '2024-02-20', '2024-03-10', '2024-04-05'],
    'Value': [100, 200, 150, 300]
})
print("Original:")
print(df_dates)
print(f"\nDtypes: {df_dates.dtypes}")

df_dates['Date'] = pd.to_datetime(df_dates['Date'])
print("\nAfter conversion:")
print(df_dates)
print(f"\nDtypes: {df_dates.dtypes}")

In [ ]:
# Date components
df_dates['Year'] = df_dates['Date'].dt.year
df_dates['Month'] = df_dates['Date'].dt.month
df_dates['Day'] = df_dates['Date'].dt.day
df_dates['DayOfWeek'] = df_dates['Date'].dt.day_name()
df_dates['IsWeekend'] = df_dates['Date'].dt.dayofweek >= 5
print(df_dates)

In [ ]:
# Date arithmetic
print("Add 7 days:")
print(df_dates['Date'] + pd.Timedelta(days=7))

print("\nAdd 1 month:")
print(df_dates['Date'] + pd.DateOffset(months=1))

print("\nDays since date:")
print((pd.Timestamp.now() - df_dates['Date']).dt.days)

print("\nDate difference:")
print(df_dates['Date'].diff())

In [ ]:
# Time series resampling
ts = pd.Series(
    np.random.randn(100),
    index=pd.date_range('2024-01-01', periods=100, freq='D')
)

print("Daily data (first 5):")
print(ts.head())

print("\nMonthly mean:")
print(ts.resample('M').mean())

print("\nWeekly sum:")
print(ts.resample('W').sum())

print("\nQuarterly stats:")
print(ts.resample('Q').agg(['mean', 'std', 'min', 'max']))

In [ ]:
# Time zone handling
ts_utc = pd.Series(
    np.random.randn(5),
    index=pd.date_range('2024-01-01', periods=5, freq='D', tz='UTC')
)
print("UTC:")
print(ts_utc)

print("\nConvert to US/Eastern:")
print(ts_utc.dt.tz_convert('US/Eastern'))

print("\nConvert to Asia/Tokyo:")
print(ts_utc.dt.tz_convert('Asia/Tokyo'))

## 8. Window Functions

In [ ]:
# Create time series data
np.random.seed(42)
ts_data = pd.Series(
    np.cumsum(np.random.randn(20)) + 100,
    index=pd.date_range('2024-01-01', periods=20, freq='D')
)
print(ts_data)

In [ ]:
# Rolling window
print("Rolling mean (window=5):")
print(ts_data.rolling(window=5).mean())

print("\nRolling std (window=5):")
print(ts_data.rolling(window=5).std())

print("\nRolling with min_periods:")
print(ts_data.rolling(window=5, min_periods=1).mean())

print("\nRolling with center=True:")
print(ts_data.rolling(window=5, center=True).mean())

In [ ]:
# Rolling with custom functions
print("Rolling max:")
print(ts_data.rolling(window=5).max())

print("\nRolling with custom function:")
print(ts_data.rolling(window=5).apply(lambda x: x.max() - x.min()))

print("\nRolling quantiles:")
print(ts_data.rolling(window=5).quantile(0.75))

In [ ]:
# Expanding window
print("Expanding mean:")
print(ts_data.expanding().mean())

print("\nExpanding sum:")
print(ts_data.expanding().sum())

print("\nExpanding with min_periods:")
print(ts_data.expanding(min_periods=3).mean())

In [ ]:
# Exponentially weighted functions
print("EWM mean (alpha=0.3):")
print(ts_data.ewm(alpha=0.3).mean())

print("\nEWM with halflife:")
print(ts_data.ewm(halflife=5).mean())

print("\nEWM std:")
print(ts_data.ewm(alpha=0.3).std())

In [ ]:
# Rolling on DataFrame
df_rolling = pd.DataFrame({
    'A': np.cumsum(np.random.randn(20)) + 100,
    'B': np.cumsum(np.random.randn(20)) + 50
})

print("Rolling mean of DataFrame:")
print(df_rolling.rolling(window=5).mean())

print("\nRolling sum of specific column:")
print(df_rolling['A'].rolling(window=3).sum())

## 9. Working with Categorical Data

In [ ]:
# Create categorical data
df_cat = pd.DataFrame({
    'Product': ['A', 'B', 'C', 'A', 'B', 'C', 'A', 'B', 'C'],
    'Rating': ['Poor', 'Good', 'Excellent', 'Good', 'Poor', 'Excellent', 'Good', 'Excellent', 'Poor'],
    'Value': [10, 20, 30, 15, 25, 35, 12, 22, 32]
})
print(df_cat)

# Convert to categorical
df_cat['Rating'] = pd.Categorical(
    df_cat['Rating'],
    categories=['Poor', 'Good', 'Excellent'],
    ordered=True
)
print(f"\nCategories: {df_cat['Rating'].cat.categories.tolist()}")
print(f"Ordered: {df_cat['Rating'].cat.ordered}")
print(df_cat)

In [ ]:
# Categorical operations
print("Value counts:")
print(df_cat['Rating'].value_counts())

print("\nMin and Max:")
print(f"Min: {df_cat['Rating'].min()}")
print(f"Max: {df_cat['Rating'].max()}")

print("\nComparisons:")
print(df_cat['Rating'] > 'Good')

print("\nGroup by categorical:")
print(df_cat.groupby('Rating', observed=True)['Value'].mean())

In [ ]:
# Modify categories
print("Add category:")
df_cat['Rating'] = df_cat['Rating'].cat.add_categories(['Outstanding'])
print(df_cat['Rating'].cat.categories.tolist())

print("\nRename categories:")
df_cat['Rating'] = df_cat['Rating'].cat.rename_categories({
    'Poor': 'Low',
    'Good': 'Medium',
    'Excellent': 'High'
})
print(df_cat['Rating'].cat.categories.tolist())

print("\nRemove categories:")
df_cat['Rating'] = df_cat['Rating'].cat.remove_categories(['Outstanding'])
print(df_cat['Rating'].cat.categories.tolist())

In [ ]:
# Convert to codes
df_cat['Rating_Code'] = df_cat['Rating'].cat.codes
print(df_cat)

# Set category order
df_cat['Rating'] = pd.Categorical(
    df_cat['Rating'],
    categories=['Low', 'Medium', 'High'],
    ordered=True
)
print("\nSorted by Rating:")
print(df_cat.sort_values('Rating'))

## 10. Handling Missing Data (Interpolation)

In [ ]:
# Create data with missing values
df_interp = pd.DataFrame({
    'A': [1, np.nan, 3, np.nan, 5, np.nan, 7],
    'B': [10, 20, np.nan, 40, 50, np.nan, 70],
    'C': [100, np.nan, np.nan, 400, 500, np.nan, 700]
})
print(df_interp)

In [ ]:
# Linear interpolation
print("Linear interpolation:")
print(df_interp.interpolate(method='linear'))

print("\nTime interpolation (requires datetime index):")
df_time = df_interp.copy()
df_time.index = pd.date_range('2024-01-01', periods=7)
print(df_time.interpolate(method='time'))

print("\nIndex interpolation:")
print(df_interp.interpolate(method='index'))

In [ ]:
# Polynomial interpolation
print("Polynomial interpolation (order=2):")
print(df_interp.interpolate(method='polynomial', order=2))

print("\nSpline interpolation:")
print(df_interp.interpolate(method='spline', order=3))

print("\nPad interpolation:")
print(df_interp.interpolate(method='pad'))

print("\nBackward fill interpolation:")
print(df_interp.interpolate(method='bfill'))

In [ ]:
# Limit interpolation
print("Limit to 1 consecutive NaN:")
print(df_interp.interpolate(method='linear', limit=1))

print("\nLimit direction forward:")
print(df_interp.interpolate(method='linear', limit_direction='forward'))

print("\nLimit direction both:")
print(df_interp.interpolate(method='linear', limit_direction='both'))

print("\nInterpolate only inside values:")
print(df_interp.interpolate(method='linear', limit_area='inside'))

print("\nInterpolate only outside values:")
print(df_interp.interpolate(method='linear', limit_area='outside'))

In [ ]:
# Interpolation with time series
ts_missing = pd.Series(
    [10, np.nan, np.nan, 40, 50, np.nan, 70, 80],
    index=pd.date_range('2024-01-01', periods=8, freq='D')
)
print("Original:")
print(ts_missing)

print("\nTime interpolation:")
print(ts_missing.interpolate(method='time'))

print("\nTime interpolation with limit:")
print(ts_missing.interpolate(method='time', limit=1))

## Practice Problems

Solve the following problems to reinforce your understanding of intermediate Pandas techniques.

In [ ]:
# Problem 1: MultiIndex and hierarchical data
# Create a MultiIndex DataFrame with the following structure:
# Outer index: ['X', 'X', 'Y', 'Y', 'Z', 'Z']
# Inner index: ['A', 'B', 'A', 'B', 'A', 'B']
# Columns: ['Sales', 'Profit']
# Values: Random integers between 100 and 500
# Then:
# a) Display the DataFrame
# b) Select all data for outer index 'Y'
# c) Select the row where outer='X' and inner='A'
# d) Unstack the DataFrame
# e) Stack it back

# Your code here:


In [ ]:
# Problem 2: Advanced GroupBy
# Create a sales DataFrame with 20 rows:
# Product: random from ['Alpha', 'Beta', 'Gamma']
# Region: random from ['North', 'South', 'East', 'West']
# Sales: random integers 100-500
# Quantity: random integers 1-50
# Then:
# a) Use agg() to compute sum, mean, and count of Sales by Product
# b) Use transform() to add a column with each product's sales percentage
# c) Use filter() to keep only products with more than 5 records
# d) Use apply() to create a summary DataFrame with total sales and average quantity per product

# Your code here:


In [ ]:
# Problem 3: Merging and joining
# Create two DataFrames:
# df_customers: ID (1-5), Name, Email
# df_orders: OrderID (1-6), CustomerID, Product, Amount
# (Note: CustomerID 6 doesn't exist in customers)
# Then:
# a) Perform an inner join
# b) Perform a left join
# c) Perform an outer join
# d) Find all orders where the customer is unknown
# e) Calculate total amount per customer

# Your code here:


In [ ]:
# Problem 4: Pivot tables
# Create a sales dataset with:
# Date: 12 monthly dates from 2024-01-01
# Product: ['A', 'B', 'C'] * 4
# Region: ['North', 'South', 'East', 'West'] * 3
# Revenue: random 1000-10000
# Units: random 10-100
# Then:
# a) Create a pivot table showing total revenue by Product and Region
# b) Add margins (totals)
# c) Create a pivot table with multiple aggregation functions
# d) Create a cross-tabulation of Product vs Region with revenue as values

# Your code here:


In [ ]:
# Problem 5: Text data processing
# Create a DataFrame with customer information:
# Name: ['john doe', 'JANE SMITH', 'Bob Johnson', 'ALICE BROWN']
# Email: ['john@company.com', 'jane@test.org', 'bob@gmail.com', 'alice@work.net']
# Phone: ['555-0101', '555-0202', '555-0303', '555-0404']
# Then:
# a) Convert names to title case
# b) Extract the domain from each email
# c) Extract the area code from each phone number
# d) Create a new column 'Username' with the part before '@' in email
# e) Filter rows where the email domain is 'company.com'

# Your code here:


In [ ]:
# Problem 6: Date and time operations
# Create a time series DataFrame with:
# Date: 30 daily dates starting from 2024-01-01
# Value: random integers 100-500
# Then:
# a) Add columns for Year, Month, Day, and Day of Week
# b) Add a column indicating whether it's a weekend
# c) Resample to weekly frequency and compute mean
# d) Resample to monthly frequency and compute sum
# e) Add 7 days to all dates and compute the difference

# Your code here:


In [ ]:
# Problem 7: Window functions
# Create a time series with 50 data points:
# Index: 50 daily dates from 2024-01-01
# Values: cumulative sum of random normal values + 100
# Then:
# a) Compute the 7-day rolling mean
# b) Compute the 7-day rolling standard deviation
# c) Compute the expanding mean
# d) Compute the exponentially weighted mean (alpha=0.2)
# e) Plot the original data and the rolling mean on the same chart

# Your code here:


In [ ]:
# Problem 8: Categorical and interpolation
# Create a DataFrame with:
# Category: ['Low', 'Medium', 'High'] repeated 5 times each
# Value: [10, np.nan, 30, np.nan, 50, np.nan, 70, np.nan, 90, np.nan, 110, np.nan, 130, np.nan, 150]
# Then:
# a) Convert Category to an ordered categorical
# b) Interpolate the Value column using linear method
# c) Sort by Category and display
# d) Group by Category and compute mean of Value
# e) Use query() to filter rows where Category is 'High' or 'Medium'

# Your code here:


## Summary

In this notebook, we covered intermediate Pandas techniques:
- **MultiIndex**: Creating and manipulating hierarchical indices
- **Advanced indexing**: Using `query()`, `isin()`, `between()`, `where()`, and `mask()`
- **Advanced GroupBy**: Using `agg()`, `transform()`, `filter()`, and `apply()`
- **Merging and joining**: SQL-style joins, concatenation, and updates
- **Pivot tables**: Creating summary tables with multiple aggregation functions
- **Text data**: Using the `str` accessor for string manipulation
- **Date and time**: Creating date ranges, converting types, and resampling
- **Window functions**: Rolling, expanding, and exponentially weighted functions
- **Categorical data**: Creating, ordering, and manipulating categorical variables
- **Interpolation**: Advanced missing data handling with various interpolation methods

Next: Move on to the Advanced notebook for performance optimization and working with large datasets.